In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
import xgboost as xgb

print(xgb.__version__)

Importing plotly failed. Interactive plots will not work.


3.4.1


In [2]:
colonnes = ["Date de début", "code site", "valeur", "validité"]

def charger(chemin, stations):
    df = pd.read_csv(chemin, sep=";", usecols=colonnes)
    df = df[df["code site"].isin(stations)]
    df = df[(df["validité"] == 1) & (df["valeur"] >= 0)]
    df = df.assign(date=pd.to_datetime(df["Date de début"]))
    return df.groupby("date")["valeur"].mean().resample("D").mean()

no2 = charger("../data/raw/geodair/no2_paris_a1_01_2015_11_2026.csv", ["FR04004", "FR04014", "FR04037", "FR04060"])
pm25 = charger("../data/raw/geodair/pm25_paris_a1_01_2015_11_2026.csv", ["FR04004", "FR04055", "FR04143"])

print(no2.shape, pm25.shape, no2.isna().sum(), pm25.isna().sum())

(4345,) (4345,) 0 143


In [3]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

def prophet_walk_forward(serie):
    df = serie.reset_index()
    df.columns = ["ds", "y"]
    morceaux = []
    for annee in range(2018, 2026):
        debut = f"{annee}-01-01"
        fin = f"{annee + 1}-01-01" if annee < 2025 else "2100-01-01"
        train = df[df["ds"] < debut]
        cible = df[(df["ds"] >= debut) & (df["ds"] < fin)]
        prevision = Prophet().fit(train).predict(cible[["ds"]])
        morceaux.append(prevision[["ds", "yhat"]])
    return pd.concat(morceaux).set_index("ds")["yhat"]

In [4]:
prophet_no2 = prophet_walk_forward(no2)
prophet_pm25 = prophet_walk_forward(pm25)

print(np.abs(no2.loc["2025":] - prophet_no2.loc["2025":]).mean())
print(np.abs(pm25.loc["2025":] - prophet_pm25.loc["2025":]).mean())

/home/bleam/code/jedha/air-quality-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
17:39:32 - cmdstanpy - INFO - Chain [1] start processing
17:39:33 - cmdstanpy - INFO - Chain [1] done processing
17:39:33 - cmdstanpy - INFO - Chain [1] start processing
17:39:33 - cmdstanpy - INFO - Chain [1] done processing
17:39:33 - cmdstanpy - INFO - Chain [1] start processing
17:39:33 - cmdstanpy - INFO - Chain [1] done processing
17:39:33 - cmdstanpy - INFO - Chain [1] start processing
17:39:33 - cmdstanpy - INFO - Chain [1] done processing
17:39:34 - cmdstanpy - INFO - Chain [1] start processing
17:39:34 - cmdstanpy - INFO - Chain [1] done processing
17:39:34 - cmdstanpy - INFO - Chain [1] start processing
17:39:34 - cmdstanpy - INFO - Chain [1] done processing
17:39:35 - cmdstanpy - INFO - Chain 

6.245856721823144
4.681709919555509


In [5]:
def creer_features(serie, prophet):
    df = pd.DataFrame({"y": serie, "prophet": prophet})
    df["hier"] = df["y"].shift(1)
    df["avant_hier"] = df["y"].shift(2)
    df["moy_7j"] = df["y"].shift(1).rolling(7).mean()
    df["residu_hier"] = (df["y"] - df["prophet"]).shift(1)
    df["jour_semaine"] = df.index.day_of_week
    df["mois"] = df.index.month
    return df.dropna(subset=["y", "prophet"])

data_no2 = creer_features(no2, prophet_no2)
data_pm25 = creer_features(pm25, prophet_pm25)
data_no2.head()

,y,prophet,hier,avant_hier,moy_7j,residu_hier,jour_semaine,mois
2018-01-01,15.643056,33.469235,13.041667,19.281597,25.575595,NaN,0,1
2018-01-02,31.103819,36.733856,15.643056,13.041667,25.067460,-17.826179,1,1
2018-01-03,18.051042,37.153488,31.103819,15.643056,26.069196,-5.630037,2,1
2018-01-04,25.818403,39.957753,18.051042,31.103819,25.038145,-19.102446,3,1
2018-01-05,40.762500,39.995576,25.818403,18.051042,22.817708,-14.139350,4,1


In [6]:
features = ["hier", "avant_hier", "moy_7j", "jour_semaine", "mois"]

def comparer(data):
    train = data.loc[:"2024"]
    test = data.loc["2025":]

    xgb_seul = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05)
    xgb_seul.fit(train[features], train["y"])
    pred_seul = xgb_seul.predict(test[features])

    features_h = features + ["residu_hier"]
    xgb_hybride = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05)
    xgb_hybride.fit(train[features_h], train["y"] - train["prophet"])
    pred_hybride = test["prophet"] + xgb_hybride.predict(test[features_h])

    resultats = {"Persistance": test["hier"], "Prophet": test["prophet"], "XGBoost seul": pred_seul, "Hybride": pred_hybride}
    for nom, pred in resultats.items():
        erreur = test["y"] - pred
        print(f"{nom:13} MAE {np.abs(erreur).mean():.2f} | RMSE {np.sqrt((erreur ** 2).mean()):.2f}")

In [7]:
print("NO2")
comparer(data_no2)
print("\nPM2.5")
comparer(data_pm25)

NO2
Persistance   MAE 5.71 | RMSE 7.70
Prophet       MAE 6.25 | RMSE 8.33
XGBoost seul  MAE 5.19 | RMSE 6.86
Hybride       MAE 4.87 | RMSE 6.61

PM2.5
Persistance   MAE 3.13 | RMSE 4.68
Prophet       MAE 4.68 | RMSE 6.77
XGBoost seul  MAE 3.08 | RMSE 4.43
Hybride       MAE 3.04 | RMSE 4.42


## Résultats (test 2025-2026, sans météo)
| Modèle | NO2 MAE | NO2 RMSE | PM2.5 MAE | PM2.5 RMSE |
|---|---|---|---|---|
| Persistance | 5.71 | 7.70 | 3.13 | 4.68 |
| Prophet | 6.25 | 8.34 | 4.68 | 6.76 |
| XGBoost seul | 5.19 | 6.86 | 3.08 | 4.43 |
| Hybride Prophet + XGBoost | 4.86 | 6.57 | 3.06 | 4.45 |

- Prophet en walk-forward annuel (2018-2025) pour des résidus hors échantillon
- Features : hier, avant-hier, moyenne 7 jours, résidu Prophet d'hier, jour de la semaine, mois
- NO2 : -15 % de MAE vs persistance. PM2.5 : -2 % seulement → la météo est nécessaire
- Hyperparamètres non optimisés (300 arbres, profondeur 3, learning rate 0.05)
